# Roman Urdu Offensive Language Detection
## Local Qwen pilot: visual results

**Model:** Qwen 3.5 4B via Ollama ? **Data:** RUHSOLD Task 1 ? **Pilot:** 30 balanced test comments (15 per class) ? **Decoding:** temperature 0

We compare zero-shot, four-example few-shot, and Cross-Script Alignment Prompting (CSAP) with 5 or 15 lexical mappings. The pilot is exploratory: 30 examples are too few for a strong generalization claim, and mappings need Urdu-speaker review.

This notebook uses self-contained SVG charts so it renders without installing plotting packages. Run all cells from the project root or `testing/`.

## Load and validate saved predictions

All plots below are generated from the saved per-example output, not hard-coded scores. The summary JSON is also loaded for run metadata.

In [1]:
from pathlib import Path
import json, csv, math
from collections import defaultdict
from IPython.display import display, HTML

ROOT = Path.cwd()
if not (ROOT / "testing/results/ollama_prompt_predictions.jsonl").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "testing/results"
rows = [json.loads(line) for line in (RESULTS / "ollama_prompt_predictions.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
summary = json.loads((RESULTS / "ollama_prompt_summary.json").read_text(encoding="utf-8"))
conditions = ["zero_shot", "few_shot", "alignment_n5", "alignment_n15"]
labels = [0, 1]
label_names = {0: "Abusive", 1: "Not abusive"}
valid = [r for r in rows if "prediction" in r]
assert all(sum(r["condition"] == c for r in valid) == summary["n_examples"] for c in conditions)
print(f"{summary['model']} ? {summary['n_examples']} examples ? {len(valid)} valid predictions ? {len(rows)-len(valid)} failed")


qwen3.5:4b ? 30 examples ? 120 valid predictions ? 13 failed


## Headline comparison

Macro-F1 is the primary metric because the task has two classes and we want to weigh them equally. Accuracy is included as a familiar secondary measure.

In [2]:
names = {"zero_shot":"Zero-shot", "few_shot":"Few-shot", "alignment_n5":"CSAP N=5", "alignment_n15":"CSAP N=15"}
metrics = {}
for c in conditions:
    rr = [r for r in valid if r["condition"] == c]
    tp = sum(r["prediction"] == r["gold"] for r in rr)
    f1s=[]
    for lab in labels:
        tp_l=sum(r["gold"]==lab and r["prediction"]==lab for r in rr)
        fp=sum(r["gold"]!=lab and r["prediction"]==lab for r in rr)
        fn=sum(r["gold"]==lab and r["prediction"]!=lab for r in rr)
        pr=tp_l/(tp_l+fp) if tp_l+fp else 0
        rec=tp_l/(tp_l+fn) if tp_l+fn else 0
        f1s.append(2*pr*rec/(pr+rec) if pr+rec else 0)
    metrics[c]={"accuracy":tp/len(rr),"macro_f1":sum(f1s)/2,"class_f1":f1s,"rows":rr}

W,H=820,390; left,top=80,48; plotw,ploth=690,260
colors=["#8496a8","#3984c6","#16a085","#8e6ac8"]
svg=[f'<svg viewBox="0 0 {W} {H}" role="img" aria-label="Accuracy and macro F1 by prompt condition" style="width:100%;max-width:900px;font-family:Arial">']
for tick in range(0,101,20):
 y=top+ploth-(tick/100)*ploth
 svg.append(f'<line x1="{left}" y1="{y}" x2="{left+plotw}" y2="{y}" stroke="#dce3ea"/><text x="{left-12}" y="{y+5}" text-anchor="end" fill="#52606d" font-size="13">{tick}%</text>')
for i,c in enumerate(conditions):
 center=left+(i+.5)*plotw/4
 for j,key in enumerate(("accuracy","macro_f1")):
  val=metrics[c][key]; bh=val*ploth; x=center-38+j*40; y=top+ploth-bh
  svg.append(f'<rect x="{x}" y="{y}" width="34" height="{bh}" rx="4" fill="{colors[i] if j==0 else "#f2a541"}"><title>{names[c]} {key}: {val:.4f}</title></rect>')
  svg.append(f'<text x="{x+17}" y="{y-7}" text-anchor="middle" font-size="11" fill="#25313c">{val:.3f}</text>')
 svg.append(f'<text x="{center}" y="{top+ploth+27}" text-anchor="middle" font-size="13" fill="#25313c">{names[c]}</text>')
svg.append(f'<text transform="translate(22 {top+ploth/2}) rotate(-90)" text-anchor="middle" fill="#52606d" font-size="13">Score</text>')
svg.append('</svg>')
legend='<div style="display:flex;gap:22px;font:13px Arial;color:#374151"><span>? Condition color: accuracy</span><span style="color:#f2a541">? Macro-F1</span></div>'
display(HTML(legend+''.join(svg)))


## Per-class F1 and confusion matrices

Rows are gold labels and columns are predicted labels in `[Abusive, Not abusive]` order. This helps show whether a gain hides a drop on one class.

In [3]:
W,H=900,350
svg=[f'<svg viewBox="0 0 {W} {H}" role="img" aria-label="Confusion matrices and per-class F1" style="width:100%;max-width:950px;font-family:Arial">']
for i,c in enumerate(conditions):
 rr=metrics[c]["rows"]
 cm=[[sum(r["gold"]==g and r["prediction"]==p for r in rr) for p in labels] for g in labels]
 ox=35+i*220; oy=55
 svg.append(f'<text x="{ox+80}" y="{oy-18}" text-anchor="middle" font-size="14" font-weight="bold">{names[c]}</text>')
 for g in range(2):
  for p in range(2):
   x=ox+52+p*58; y=oy+g*48; v=cm[g][p]; shade=220-min(130,v*9)
   svg.append(f'<rect x="{x}" y="{y}" width="54" height="44" rx="3" fill="rgb(40,{shade},150)"/>')
   svg.append(f'<text x="{x+27}" y="{y+28}" text-anchor="middle" fill="white" font-size="16" font-weight="bold">{v}</text>')
 svg.append(f'<text x="{ox+80}" y="{oy+119}" text-anchor="middle" font-size="10" fill="#536170">gold ? / predicted ?</text>')
 for j,lab in enumerate(labels):
  val=metrics[c]["class_f1"][j]; x=ox+13+j*100; y=oy+165; bw=82*val
  svg.append(f'<text x="{x}" y="{y-8}" font-size="11">{label_names[lab]} {val:.2f}</text>')
  svg.append(f'<rect x="{x}" y="{y}" width="82" height="12" rx="5" fill="#e5e7eb"/><rect x="{x}" y="{y}" width="{bw}" height="12" rx="5" fill="#16a085"/>')
svg.append('</svg>')
display(HTML(''.join(svg)))


## Latency and run health

Latency is the mean wall-clock time per local request. It is hardware- and prompt-length-dependent; this is descriptive for this machine/model run, not a general speed benchmark.

In [4]:
W,H=760,270; left,top=100,35; plotw,ploth=570,175
lat=[sum(r["latency_seconds"] for r in metrics[c]["rows"])/len(metrics[c]["rows"]) for c in conditions]
mx=max(lat)*1.2
svg=[f'<svg viewBox="0 0 {W} {H}" role="img" aria-label="Mean inference latency" style="width:100%;max-width:800px;font-family:Arial">']
for i,c in enumerate(conditions):
 bw=86; x=left+i*137; bh=lat[i]/mx*ploth; y=top+ploth-bh
 svg.append(f'<rect x="{x}" y="{y}" width="{bw}" height="{bh}" rx="4" fill="#3984c6"/><text x="{x+bw/2}" y="{y-8}" text-anchor="middle" font-size="12">{lat[i]:.2f}s</text><text x="{x+bw/2}" y="{top+ploth+22}" text-anchor="middle" font-size="12">{names[c]}</text>')
svg.append('</svg>')
display(HTML(''.join(svg)))


## Interpretation

In this 30-example balanced pilot, few-shot prompting improved over zero-shot, while both alignment conditions scored higher on macro-F1. N=15 was only slightly above N=5. The sample is too small to conclude that alignment generalizes; the lexicon and demonstrations should be reviewed and frozen, then evaluated on a larger predeclared set.

**Reported pilot scores:** zero-shot 0.8316; few-shot 0.8643; CSAP N=5 0.8990; CSAP N=15 0.8999 macro-F1. Full rows are in `testing/results/ollama_prompt_predictions.jsonl`.